In [ ]:
# Dataset and data root config

dataset = 'mlc25'
data_root = './data'
use_optuna = True

In [ ]:
import numpy as np

def mean_euclidean_error(y_true, y_pred):
    # Calculate the difference between true and predicted values
    diff = y_true - y_pred
    
    # Reshapes row vectors to column vectors for consistent norm calculation
    if diff.ndim == 1:
        diff = diff.reshape(-1, 1)

    # Calculate mean Euclidean error
    return np.linalg.norm(diff, axis=1).mean()

In [ ]:
from sklearn.metrics import make_scorer

# Create a scorer for use in model evaluation
mee_scorer = make_scorer(mean_euclidean_error, greater_is_better=False)

### Data Loading

If you occur in "SyntaxError: f-string: unmatched '(' (keras.py, line 50)" just run again the cell

In [ ]:
# === 2. IMPORTS AND UTILITIES ===
from utils.data_loader import get_monk1_data, get_ml_cup_data  
from torch.utils.data import DataLoader, TensorDataset
from models import SVCModel, SVRModel
from sklearn.metrics import *
from sklearn.multioutput import MultiOutputRegressor
  
    
# --- Utility function to extract data from DataLoader to NumPy arrays ---
def extract_data_to_numpy(data_loader):
    """
    Converts data from a PyTorch DataLoader into a flattened NumPy array pair (X, y).
    Targets (y) are returned in their original dimensionality (e.g., [N, M] for multi-output).
    """
    X_list = []
    y_list = []
    for X, y in data_loader:
        # Flatten the input (e.g., 28x28 image -> 784 features)
        X_list.append(X.view(X.size(0), -1).numpy()) 
        # Convert labels to NumPy
        y_list.append(y.numpy())
    
    X_data = np.concatenate(X_list)
    y_data = np.concatenate(y_list)
    
    # We return y_data as is (2D array, e.g., [N, 1] or [N, M]).
    return X_data, y_data

In [ ]:
from sklearn.preprocessing import StandardScaler

# Use StandardScaler for feature scaling with 0 mean and unit variance
scaler = StandardScaler()

In [ ]:
# === 3. DATA LOADING AND PREPARATION ===

from utils.data_loader import get_monk1_data, get_ml_cup_data
import sys

dataset_name = dataset
BATCH_SIZE = 1024 # Batch size for DataLoader (not used in SVM but for consistency) 

print(f"Loading dataset: {dataset_name.upper()}...")
    
# Determine task type and load data (DataLoader objects are returned)
if dataset_name == 'monk1':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk1_data(BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"

elif dataset_name == 'mlc25':
    train_loader, test_loader = get_ml_cup_data(BATCH_SIZE, data_root, test_ratio = 0.25, mps = False,scaler = scaler)
    is_regression_task = True
    metric_name = "Test MEE"

else:
    # This block handles the error if the dataset is outside the specified choices (monk1, mlc25).
    print("Unsupported dataset for SVM.")
    sys.exit(1)


# --- 3.2 Data Preparation for Scikit-learn ---

# Convert DataLoaders (PyTorch) to NumPy arrays (Scikit-learn)
X_train, y_train = extract_data_to_numpy(train_loader)
X_test, y_test = extract_data_to_numpy(test_loader)

print(f"Data loaded: Training samples={X_train.shape[0]}, Test samples={X_test.shape[0]}")

In [ ]:
# Verify shapes

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
if not use_optuna:
    from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold
    from sklearn.svm import SVC, SVR
    from scipy.stats import loguniform

    # 1. Definition of parameters in common for both SVC and SVR
    common_params = [
        # 1. Linear Kernel: C is the ONLY parameter. Gamma is implicitly 'scale' or ignored.
        {
            'kernel': ['linear'],
            'C': loguniform(1e-3, 1e2),
        },
        
        # 2. RBF/Poly Kernels with Discrete Gamma: Tests the two known heuristics.
        {
            'kernel': ['rbf', 'poly'],
            'C': loguniform(1e-3, 1e2),
            'gamma': ['scale', 'auto'], # Discrete strings only
        },
        
        # 3. RBF/Poly Kernels with Continuous Gamma: Randomly samples from the continuous range.
        {
            'kernel': ['rbf', 'poly'],
            'C': loguniform(1e-3, 1e2),
            'gamma': loguniform(1e-3, 1e1), # Continuous distribution object only
        },
    ]

    # 2. Conditional logic for epsilon parameter in SVR
    if is_regression_task:
        params = []
        for p in common_params:
            p_new = p.copy()
            p_new['epsilon'] = loguniform(1e-2,1e1)
            params.append(p_new)  # Append modified copy for SVR
        
        print("SVR parameters with epsilon added.")

    else:
        params = common_params
        print("SVC parameters without epsilon.")


    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) if not is_regression_task else KFold(n_splits=5, shuffle=True, random_state=42)  

In [ ]:
if not use_optuna:
    # HP Randomized Search Configuration

    limit = 100000
    size = 1000

    # Max iter and cache size set for efficiency
    base_estimator = SVC(max_iter=limit, cache_size=size) if not is_regression_task \
                else SVR(max_iter=limit, cache_size=size)

    hp_search = RandomizedSearchCV(
        estimator = base_estimator,
        param_distributions = params,
        n_iter = 200,                     # Number of random configurations to try
        scoring = 'accuracy' if not is_regression_task else mee_scorer,
        cv = kfold, 
        n_jobs = -1,
        verbose = 1,
        random_state = 42
     )                          


In [ ]:
if not use_optuna:
    if is_regression_task:
        # Wrap in MultiOutputRegressor if the dataset is mlc25
        final_model = MultiOutputRegressor(hp_search)  
    else:
        final_model = hp_search

In [ ]:
if not use_optuna:
    final_model.fit(X_train, y_train)

    best_cv_scores = []

    if is_regression_task:
        for estimator in final_model.estimators_:
            best_cv_scores.append(abs(estimator.best_score_))
    else:
        best_cv_scores.append(final_model.best_score_)

In [ ]:
if not use_optuna:
    if is_regression_task:
        # Print estimators for each output if mlc25
        print(final_model.estimators_)
    else:
        print(final_model.best_params_)
        

In [ ]:
if not use_optuna:
        
    # === 4. MODEL INITIALIZATION (FINAL RECONSTRUCTION) ===

    # --- 4.0 Helper Function to Create Wrapper from Estimator ---
    def create_model_from_estimator(best_estimator):
        """
        Extracts parameters from the best_estimator and creates an instance
        of the following custom wrapper (SVCModel or SVRModel).
        """
        # 1. Get parameters from the fitted model
        params = best_estimator.get_params()
        
        # 2. Determine the type and instantiate the correct wrapper
        if isinstance(best_estimator, SVC):
            return SVCModel(**params), 'svc'
        elif isinstance(best_estimator, SVR):
            return SVRModel(**params), 'svr'
        else:
            raise TypeError(f"Type unsupported for reconstruction: {type(best_estimator)}")

In [ ]:
if not use_optuna:
    # --- 4.1 Reconstruction Logic ---

    final_models_list = []  # List to hold final model wrappers
    is_multi_output = False # Flag to indicate multi-output scenario

    # MLC25 case: check if final_model has 'estimators_' attribute typical of MultiOutput wrapper
    if hasattr(final_model, 'estimators_'):
        print(f"Detected Multi-Output System ({len(final_model.estimators_)} targets).")
        is_multi_output = True
        
        # Iterate over each RandomizedSearchCV contained in the MultiOutput
        for i, search_obj in enumerate(final_model.estimators_):
            # Extract the winner for this specific target
            best_est = search_obj.best_estimator_
            
            # Create wrapper
            wrapper, m_type = create_model_from_estimator(best_est)
            final_models_list.append(wrapper)
            
            print(f"Target {i}: Configured {m_type.upper()} with C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}, epsilon={getattr(wrapper.model, 'epsilon', 'N/A')}")

    # MONK1 case: Single Output
    else:
        print("Detected Single-Output System.")
        # final_model is directly the RandomizedSearchCV
        best_est = final_model.best_estimator_
        best_params = best_est.get_params()

        best_params['probability'] = True  
        
        wrapper, m_type = create_model_from_estimator(best_est)
        final_models_list.append(wrapper)
        
        print(f"Configured {m_type.upper()} with C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}")

In [ ]:
if use_optuna:
    # === 4. HYPERPARAMETER OPTIMIZATION (OPTUNA) ===
    import optuna
    import warnings
    from sklearn.svm import SVR, SVC
    from sklearn.model_selection import cross_val_score
    from optuna.samplers import TPESampler

    # List to hold final model wrappers
    final_models_list = [] 
    best_cv_scores = [] 
    N_TRIALS = 200

    print(f"🚀 Starting Optuna Optimization ({N_TRIALS} trials)...")

    # --- ML-CUP ---
    if is_regression_task:
        print(">>> Optimizing Multi-Output SVR (One model per target)...")
        
        # Loop over each target for multi-output regression
        for i in range(y_train.shape[1]):
            print(f"\n--- Optimizing Target {i} ---")
            y_current = y_train[:, i]
            
            def objective_svr(trial):
                # Kernel choice
                kernel_choice = trial.suggest_categorical('kernel', ['rbf','poly','linear'])

                # Conditional parameters
                gamma_param = 'scale'  # Default for linear kernel
                degree_param = 3
                coef0_param = 0.0

                if kernel_choice == 'poly':
                    gamma_param = trial.suggest_float('gamma', 1e-3, 1e1, log=True)
                    degree_param = trial.suggest_int('degree', 2, 5)
                    coef0_param = trial.suggest_float('coef0', 0.0, 10.0)
                elif kernel_choice == 'rbf':
                    gamma_param = trial.suggest_float('gamma', 1e-3, 1e1, log=True)

                # 1. Search Space Definition
                param = {
                    'kernel': kernel_choice,
                    'C': trial.suggest_float('C', 1e-3, 1e2, log=True),       
                    'epsilon': trial.suggest_float('epsilon', 1e-2, 1e1, log=True), 
                    'gamma': gamma_param,
                    'degree': degree_param,
                    'coef0': coef0_param
                }
                
                # 2. Model Instantiation
                model = SVR(**param, cache_size=1000, max_iter=100000)
                
                # 3. Validation  
                scores = cross_val_score(model, X_train, y_current, cv=5, scoring=mee_scorer, n_jobs=-1)
                return scores.mean() 

            # --- OPTUNA STUDY EXECUTION ---
            study = optuna.create_study(direction="maximize")
            study.optimize(objective_svr, n_trials=N_TRIALS, show_progress_bar=False)
            
            best_cv_score = abs(study.best_value)
            best_cv_scores.append(best_cv_score)

            print(f"✅ Best Params T{i}: {study.best_params}")
            print(f"📉 Best CV Score (MEE): {-study.best_value:.4f}")  
            
            # --- WRAPPER CREATION ---
            best_wrapper = SVRModel(**study.best_params, cache_size=1000, max_iter=100000)
            best_wrapper.fit(X_train, y_current) 
            
            final_models_list.append(best_wrapper)
            is_multi_output = True

    # --- MONK1 ---
    else:
        print(">>> Optimizing SVC (Single Output)...")
        
        def objective_svc(trial):
            kernel_choice = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])

            degree_param = 3
            coef0_param = 0.0
            gamma_param = 'scale'

            if kernel_choice == 'poly':
                degree_param = trial.suggest_int('degree', 2, 5)
                coef0_param = trial.suggest_float('coef0', 0.0, 10.0)
            elif kernel_choice != 'linear':
                gamma_param = trial.suggest_float('gamma', 1e-3, 1e1, log=True)

            param = {
                'kernel': kernel_choice,
                'C': trial.suggest_float('C', 1e-3, 1e2, log=True),
            }

            if kernel_choice != 'linear':
                param['gamma'] = gamma_param
            if kernel_choice == 'poly':
                param['degree'] = degree_param
                param['coef0'] = coef0_param
                
            model = SVC(**param, cache_size=1000)
            scores = cross_val_score(model, X_train, y_train.ravel(), cv=5, scoring='accuracy', n_jobs=-1)
            return scores.mean()

        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
        study.optimize(objective_svc, n_trials=N_TRIALS, show_progress_bar=True)
        
        best_cv_scores.append(study.best_value)

        print(f"✅ Best Params: {study.best_params}")
        print(f"📈 Best Accuracy: {study.best_value:.2%}")
        
        # --- WRAPPER CREATION---
        best_wrapper = SVCModel(**study.best_params, cache_size=1000, probability=True)
        best_wrapper.fit(X_train, y_train.ravel())
        
        final_models_list.append(best_wrapper)
        is_multi_output = False  # Flag for single-output
        
    print(f"\n🎉 Optimization Complete. Models ready in 'final_models_list': {len(final_models_list)}")

In [ ]:
# --- 4.2 Final Fitting ---

print("\n--- Starting Final Refitting of Models ---")

if is_multi_output:
    for i, wrapper in enumerate(final_models_list):
        print(f"Fitting Target {i}...")
        # Note: y_train[:, i] takes only the i-th column
        wrapper.fit(X_train, y_train[:, i]) 
        
else:
    print("Fitting Single Model...")
    final_models_list[0].fit(X_train, y_train.ravel())

print("Refitting done.")

In [ ]:
# === 6. PREDICTION & EVALUATION ===

# 6.1 Generation of Predictions (Handles List vs Single Model Case)
print("\n--- Generating Predictions ---")

if is_multi_output:
    # Multi-Output Case: Iterate over the list of models
    column_preds = []
    for wrapper in final_models_list:
        pred = wrapper.predict(X_test)
        column_preds.append(pred)
    
    # Combine columns: (N_samples, N_targets)
    y_test_pred = np.column_stack(column_preds)
    
    y_test_eval = y_test 
    
else:
    # Single Output Case: Directly use the single model
    y_test_pred = final_models_list[0].predict(X_test)
    
    # Flatten y_test for comparison
    y_test_eval = y_test.ravel()

# Ensure y_test is in the correct shape for comparison
print(f"Predictions generated. Shape: {y_test_pred.shape}")

In [ ]:
# 6.2 Calculating Metrics

print("\n--- Calculating Metrics ---")

if is_regression_task:
    # Calculate MEE for Regression
    final_metric = mean_euclidean_error(y_test_eval, y_test_pred)
else:
    # Calculate Accuracy for Classification
    final_metric = accuracy_score(y_test_eval, y_test_pred) * 100.0
    
# 6.3 Risultato Finale
print(f"Final {metric_name}: {final_metric}")

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import make_scorer

if is_regression_task:
    # 1. Define dummy model
    # strategy='mean' predicts always the mean of the training set for each target
    dummy_regr = DummyRegressor(strategy="mean")

    # 2. Training (just computes means, it's instantaneous)
    print("\n--- Training Baseline (Dummy Regressor) ---")
    dummy_regr.fit(X_train, y_train)

    # 3. Prediction
    y_pred_dummy = dummy_regr.predict(X_test)

    # 4. Baseline MEE Calculation
    mee_baseline = mean_euclidean_error(y_test, y_pred_dummy)

    print(f"Baseline MEE (Mean Strategy): {mee_baseline:.4f}")
    print(f"Best SVR model MEE:          {final_metric:.4f}") 

    # 5. Comparison
    if final_metric < mee_baseline:
        print("SUCCESS: the model outperforms the baseline!")
        improvement = mee_baseline - final_metric
        print(f"   Improvement: {improvement:.4f} MEE points")
    else:
        print("FAIL: the model is worse than the baseline.")

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

if not is_regression_task:
    print("\n--- Training Baseline (Dummy Classifier) ---")
    
    # 1. Define dummy model
    # strategy='most_frequent' always predicts the most frequent class in the training set
    dummy_clf = DummyClassifier(strategy="most_frequent")
    
    # 2. Training 
    dummy_clf.fit(X_train, y_train.ravel())
    
    # 3. Prediction
    y_pred_dummy = dummy_clf.predict(X_test)
    
    # 4. Baseline Accuracy Calculation
    acc_baseline = accuracy_score(y_test.ravel(), y_pred_dummy) * 100.0
    
    print(f"Baseline Accuracy (Most Frequent): {acc_baseline:.2f}%")
    print(f"Best SVC model Accuracy:          {final_metric:.2f}%") 
    
    # 5. Comparison
    if final_metric > acc_baseline:
        print("SUCCESS: the model outperforms the baseline!")
        print(f"   Improvement: +{final_metric - acc_baseline:.2f}%")
    else:
        print("FAIL: the model is worse than the baseline.")

In [ ]:
# === LEARNING CURVE VISUALIZATION ===
import matplotlib.pyplot as plt

print("\n--- Generating Learning Curves ---")

if is_regression_task:
    # Use MEE as scoring metric for regression
    score_metric = mee_scorer 
else:
    # For classification use standard accuracy
    score_metric = 'accuracy'

# Loop over models (1 for Monk, 4 for ML-CUP)
for i, wrapper in enumerate(final_models_list):
    print(f"\nGenerating curve for model {i}...")
    
    # Select appropriate target data
    if is_regression_task:
        # Choose the specific i-th column for ML-CUP
        current_y = y_train[:, i]
        title = f"Target {i} Learning Curve"
    else:
        # In case of MONK use ravel() to flatten the array
        current_y = y_train.ravel()
        title = "Monk-1 Classification Learning Curve"
        
    wrapper.plot_learning_curve(X_train, current_y, cv=5, scoring=score_metric)

print("\n--- Done ---")

In [ ]:
# === 7. SYSTEM SAVING (Updated Structure) ===
import os
import json
import datetime
import joblib

def save_model_system(models_list, scaler, dataset_name, is_regression, optimization_method, 
                      final_test_score, cv_scores_list, 
                      base_dir='./svm_saved_models', model_tag="Best_Model"):
    """
    Saves the trained models, scaler, and configuration to a structured directory 
    and updates a central JSON registry with performance metrics.
    
    Args:
        models_list: List of trained model wrappers (SVRModel or SVCModel).
        scaler: The fitted StandardScaler object.
        dataset_name: Name of the dataset (e.g., 'mlc25', 'monk1').
        is_regression: Boolean flag for regression task.
        optimization_method: String identifier (e.g., 'Optuna', 'RandomSearch').
        final_test_score: Float, the score on the Internal Test Set (MEE or Accuracy).
        cv_scores_list: List of floats, best CV scores (one per target).
        base_dir: Root directory for saving models.
        model_tag: String tag for the folder name (e.g., 'SVR_Optimized').
    """
    
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Directory Setup
    # Create structure: ./svm_saved_models/regression/Optuna/RunName/
    task_subfolder = "regression" if is_regression else "classification"
    method_folder = os.path.join(base_dir, task_subfolder, optimization_method)
    
    # Unique name for this specific run
    run_name = f"{dataset_name}_{model_tag}_{timestamp}"
    save_path_root = os.path.join(method_folder, run_name)
    os.makedirs(save_path_root, exist_ok=True)
    
    # Path to the registry file (located in the method folder)
    registry_file = os.path.join(method_folder, f"{task_subfolder}_{optimization_method}_registry.json")

    # 2. Load Existing Registry
    registry = {}
    if os.path.exists(registry_file):
        try:
            with open(registry_file, 'r') as f:
                registry = json.load(f)
        except json.JSONDecodeError:
            print("Warning: Registry file is corrupted. Starting fresh.")
    
    # 3. Save Scaler
    # Essential for ensuring future data is preprocessed exactly like training data
    scaler_path = os.path.join(save_path_root, "scaler.pkl")
    joblib.dump(scaler, scaler_path)
    print(f"📁 Scaler saved to: {scaler_path}")

    # 4. Construct JSON Entry
    entry_key = run_name
    
    json_entry = {
        "type": "MultiOutput SVR" if is_regression else "SVC",
        "method": optimization_method,
        "timestamp": timestamp,
        "location": save_path_root,
        
        # Global Score on Internal Test Set (Calculated in Cell 6.2)
        "global_test_score": round(final_test_score, 4),
        "metric_name": "MEE" if is_regression else "Accuracy",
        
        "targets": {} # Dictionary to hold details for each specific model/target
    }

    # 5. Save Models and Target Details
    if is_regression:
        # === MULTI-OUTPUT CASE (ML-CUP) ===
        for i, wrapper in enumerate(models_list):
            # A. Save the PKL file
            filename = f"target_{i}.pkl"
            full_path = os.path.join(save_path_root, filename)
            wrapper.save_pkl(full_path)
            
            # B. Populate JSON with specific CV score and params
            json_entry["targets"][f"target_{i}"] = {
                "cv_score": round(cv_scores_list[i], 4), # Best CV score for this target
                "params": wrapper.get_params_clean()
            }
    else:
        # === SINGLE-OUTPUT CASE (MONK) ===
        wrapper = models_list[0]
        
        # A. Save the PKL file
        filename = "classifier_model.pkl"
        full_path = os.path.join(save_path_root, filename)
        wrapper.save_pkl(full_path)
        
        # B. Populate JSON 
        json_entry["targets"]["classifier"] = {
            "cv_score": round(cv_scores_list[0], 4),
            "params": wrapper.get_params_clean()
        }

    # 6. Write to Disk
    registry[entry_key] = json_entry
    
    with open(registry_file, 'w') as f:
        json.dump(registry, f, indent=4)
        
    print(f"📝 Registry updated: {registry_file}")
    print(f"✅ System Saved in: {save_path_root}")


# --- FUNCTION CALL ---
# Determine parameters based on configuration
current_method = "Optuna" if use_optuna else "RandomSearch"
tag = "SVR_Optimized" if is_regression_task else "SVC_Optimized"

# Execute saving
save_model_system(
    models_list=final_models_list, 
    scaler=scaler,
    dataset_name=dataset_name, 
    is_regression=is_regression_task,
    optimization_method=current_method,
    final_test_score=final_metric,   # Passed from Cell 6.2
    cv_scores_list=best_cv_scores,   # Passed from Training Step
    model_tag=tag
)

In [ ]:
# === 9. VISUALIZATIONS (INFERENCE ANALYSIS) ===
# This section generates plots using methods integrated in the wrappers.

print("\n--- Generating Analysis Plots ---")

if is_regression_task:
    # --- ML-CUP (Regression) ---
    # Generate one scatter plot for each target (4 total)
    
    for i, wrapper in enumerate(final_models_list):
        print(f"Plotting Target {i}...")
        
        # Get the correct column of y_test for this target
        y_test_target = y_test[:, i]
        
        # Nota: Non serve passare y_pred, lo calcola lui internamente da X_test
        wrapper.plot_regression_analysis(X_test, y_test_target, title_suffix=f"Target {i}")

else:
    # --- MONK (Classification) ---
    # Generate Confusion Matrix and ROC Curve
    print("Plotting Classification Analysis...")
    
    wrapper = final_models_list[0]
    
    wrapper.plot_classification_analysis(X_test, y_test.ravel())

print("Visualizations complete.")